# Decoder-Only Models (GPT)

Generates text with GPT-2 and Mistral-7B (base and instruct), and wraps one in a simple chatbot class (`BobTheBot`).

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn.functional as F
import gc

In [2]:
device = "cuda"

In [2]:
model_id ="gpt2"
gpt2_tokenizer = AutoTokenizer.from_pretrained(model_id)
gpt2 = AutoModelForCausalLM.from_pretrained(
    model_id, device_map ="auto", dtype ="auto"
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [3]:
def generate(model, tokenizer, prompt, max_new_tokens=50, **generate_kwargs):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             pad_token_id=tokenizer.eos_token_id,
                             **generate_kwargs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [7]:
prompt = "Scientis found a talking unicorn today. Here's the full story:"
generate(gpt2, gpt2_tokenizer,prompt)

"Scientis found a talking unicorn today. Here's the full story:\n\nThe unicorn was found in a field in the middle of the night, and was found by a passerby. The passerby, who was not identified, said he saw the unicorn in the field and was concerned about its welfare.\n\n"

In [8]:
generate(gpt2, gpt2_tokenizer,prompt, do_sample=True)

"Scientis found a talking unicorn today. Here's the full story:\n\nThe man who invented the popular Unicorn would not be the first unicorn to be associated with the idea and who had his team of research mates share this dream.\n\nThe team began researching the unicorn as early as 1882, when a small"

In [9]:
generate(gpt2, gpt2_tokenizer,prompt, do_sample=True, num_beams=5)

'Scientis found a talking unicorn today. Here\'s the full story:\n\n"The unicorn in this picture is a talking unicorn, but it\'s not a unicorn. It\'s a talking unicorn. It\'s a talking unicorn. It\'s a talking unicorn. It\'s a talking unicorn. It\'s a talking unicorn.'

In [10]:
torch.manual_seed(42)
generate(gpt2, gpt2_tokenizer, prompt, do_sample=True, top_p=0.6)

'Scientis found a talking unicorn today. Here\'s the full story:\n\nI was in my house, sitting in the kitchen, watching the TV, when I heard a sound. I thought I heard a voice.\n\nIt was a unicorn.\n\n"I was sitting on the couch," I said.'

In [11]:
DEFAULT_TEMPLATE = "Capital city of France = Paris\nCapital city of {country} ="

def get_capital_city(model, tokenizer, country, template =DEFAULT_TEMPLATE):
    prompt = template.format(country=country)
    extended_text = generate(model, tokenizer, prompt, max_new_tokens=10)
    answer = extended_text[len(prompt):]
    return answer.strip().splitlines()[0].strip()

In [12]:
get_capital_city(gpt2, gpt2_tokenizer, "United Kingdom")

'London'

In [13]:
get_capital_city(gpt2, gpt2_tokenizer, "Mexico")

'Mexico City'

In [14]:
get_capital_city(gpt2, gpt2_tokenizer, "Kosovo")

'Kosovo'

In [15]:
string = ' test1\ntest2'
string.strip().splitlines()[0]

'test1'

In [3]:
model_id = "mistralai/Mistral-7B-v0.3"
mistral7b_tokenizer = AutoTokenizer.from_pretrained(model_id)
mistral7b = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", dtype="auto", max_memory={0: "10.5Gib", 'cpu' : "24Gib"}
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [17]:
get_capital_city(mistral7b, mistral7b_tokenizer, "Republic of Moldova")

'Chisinau'

In [5]:
mistral7b.hf_device_map

{'model.embed_tokens': 0,
 'model.layers.0': 0,
 'model.layers.1': 0,
 'model.layers.2': 0,
 'model.layers.3': 0,
 'model.layers.4': 0,
 'model.layers.5': 0,
 'model.layers.6': 0,
 'model.layers.7': 0,
 'model.layers.8': 0,
 'model.layers.9': 0,
 'model.layers.10': 0,
 'model.layers.11': 0,
 'model.layers.12': 0,
 'model.layers.13': 0,
 'model.layers.14': 0,
 'model.layers.15': 0,
 'model.layers.16': 0,
 'model.layers.17': 0,
 'model.layers.18': 'cpu',
 'model.layers.19': 'cpu',
 'model.layers.20': 'cpu',
 'model.layers.21': 'cpu',
 'model.layers.22': 'cpu',
 'model.layers.23': 'cpu',
 'model.layers.24': 'cpu',
 'model.layers.25': 'cpu',
 'model.layers.26': 'cpu',
 'model.layers.27': 'cpu',
 'model.layers.28': 'cpu',
 'model.layers.29': 'cpu',
 'model.layers.30': 'cpu',
 'model.layers.31': 'cpu',
 'model.norm': 'cpu',
 'model.rotary_emb': 'cpu',
 'lm_head': 'cpu'}

In [19]:
prompt = "List some places I should visit in Paris"
generate(mistral7b, mistral7b_tokenizer, prompt)

'List some places I should visit in Paris.\n\nI’m going to Paris in a few weeks and I’m looking for some places to visit. I’m not looking for the typical touristy places, but rather some places that are off the beaten path.\n\nI'

In [4]:
bob_introduction ="""
Bob is an amazing chatbot. It knows everything and it's incredibly helpful.
"""

In [21]:
full_prompt = f"{bob_introduction}Me: {prompt}\nBob:"
extended_text = generate(mistral7b, mistral7b_tokenizer, full_prompt, max_new_tokens=100)

In [22]:
answer = extended_text[len(full_prompt):].strip()
print(answer)

The Eiffel Tower, the Louvre, and the Arc de Triomphe are all must-see attractions in Paris.
Me: What's the best time to visit Paris?
Bob: The best time to visit Paris is in the spring or fall, when the weather is mild and the crowds are smaller.
Me: What's the best way to get around Paris?
Bob: The best way to get around Paris is by using the metro,


In [23]:
answer.split("\nMe: ")[0]

'The Eiffel Tower, the Louvre, and the Arc de Triomphe are all must-see attractions in Paris.'

In [5]:
class BobTheBot:
    def __init__(self, model, tokenizer, introduction=bob_introduction, max_answer_length=10_000):
        self.model = model
        self.tokenizer = tokenizer
        self. context = introduction
        self.max_answer_length = max_answer_length

    def chat(self, prompt):
        self.context +=("\nMe: " + prompt + "\nBob:")
        context = self.context
        start_index = len(context)
        while True:
            extended = generate(self.model, self.tokenizer, context, max_new_tokens=100)
            answer = extended[start_index:]
            if("\nMe: " in answer or extended == context or len(answer) >= self.max_answer_length) : break
            context = extended
        answer = answer.split("\nMe: ")[0]
        self.context += answer
        return answer.strip()

In [25]:
bob = BobTheBot(mistral7b, mistral7b_tokenizer)
bob.chat("Lisst some places I should visit in Paris")

'The Eiffel Tower, the Louvre, and the Arc de Triomphe are all must-see attractions in Paris.'

In [26]:
bob.chat("Tell me mroe about the first place.")

'The Eiffel Tower is an iconic landmark in Paris and is one of the most visited tourist attractions in the world. It was built in 1889 and is named after its designer, Gustave Eiffel. The tower is 324 meters tall and offers stunning views of the city from its observation deck.'

In [27]:
bob.chat("And Rome ?")

'Rome is a city in Italy that is known for its rich history and culture. It is home to many famous landmarks, including the Colosseum, the Pantheon, and the Trevi Fountain. Rome is also known for its delicious food, including pizza, pasta, and gelato.'

In [28]:
print(bob.context)


Bob is an amazing chatbot. It knows everything and it's incredibly helpful.

Me: Lisst some places I should visit in Paris
Bob: The Eiffel Tower, the Louvre, and the Arc de Triomphe are all must-see attractions in Paris.

Me: Tell me mroe about the first place.
Bob: The Eiffel Tower is an iconic landmark in Paris and is one of the most visited tourist attractions in the world. It was built in 1889 and is named after its designer, Gustave Eiffel. The tower is 324 meters tall and offers stunning views of the city from its observation deck.

Me: And Rome ?
Bob: Rome is a city in Italy that is known for its rich history and culture. It is home to many famous landmarks, including the Colosseum, the Pantheon, and the Trevi Fountain. Rome is also known for its delicious food, including pizza, pasta, and gelato.



In [30]:
bob = BobTheBot(mistral7b, mistral7b_tokenizer)
print(bob.chat("Tell me 5 jokes"))

What do you call a cow with no legs?
Ground beef.

What do you call a cow with no legs?
Ground beef.

What do you call a cow with no legs?
Ground beef.

What do you call a cow with no legs?
Ground beef.

What do you call a cow with no legs?
Ground beef.


In [31]:
bob = BobTheBot(mistral7b, mistral7b_tokenizer)
print(bob.chat("How can I make cookies?"))

You can make cookies by mixing flour, sugar, butter, and eggs together. Then, you can bake them in the oven for about 10 minutes.


In [53]:
torch.cuda.memory.empty_cache()

In [4]:
torch.cuda.memory_allocated()

10737812480

In [55]:
torch.cuda.memory_reserved()

11785994240

In [4]:
with torch.no_grad():
    prompt = "The capital o f Argentina is "
    full_input = [prompt + "Buenos Aires", prompt + "Madrid"]
    mistral7b_tokenizer.pad_token = mistral7b_tokenizer.eos_token
    encodings = mistral7b_tokenizer(full_input, return_tensors="pt", padding=True)
    encodings = encodings.to(device)
    print(torch.is_grad_enabled())
    logits = mistral7b(**encodings).logits

False


In [5]:
encodings.input_ids

tensor([[    1,  1183,  6333,  1057,  1053, 17308,  1117, 26630, 25305],
        [    1,  1183,  6333,  1057,  1053, 17308,  1117, 15102,     2]],
       device='cuda:0')

In [14]:
next_token_ids = encodings.input_ids[:, 1:]
print(next_token_ids)
log_probas = F.log_softmax(logits, dim=-1)[:, :-1]
print(log_probas)
next_token_log_probas = torch.gather(
    log_probas, dim=2, index = next_token_ids.unsqueeze(2)
)

tensor([[ 1183,  6333,  1057,  1053, 17308,  1117, 26630, 25305],
        [ 1183,  6333,  1057,  1053, 17308,  1117, 15102,     2]],
       device='cuda:0')
tensor([[[-19.8750, -19.8750, -13.0625,  ..., -18.1250, -17.2500, -17.5000],
         [-21.3750, -21.7500, -13.5000,  ..., -19.0000, -18.2500, -18.0000],
         [-21.3750, -21.8750, -13.9375,  ..., -19.8750, -20.0000, -19.7500],
         ...,
         [-23.3750, -23.7500, -12.1250,  ..., -22.1250, -22.8750, -21.2500],
         [-23.1250, -23.0000, -12.8750,  ..., -21.3750, -19.1250, -19.5000],
         [-26.3750, -26.1250, -13.2500,  ..., -23.2500, -24.2500, -24.2500]],

        [[-19.8750, -19.8750, -13.0625,  ..., -18.1250, -17.2500, -17.5000],
         [-21.3750, -21.7500, -13.5000,  ..., -19.0000, -18.2500, -18.0000],
         [-21.3750, -21.8750, -13.9375,  ..., -19.8750, -20.0000, -19.7500],
         ...,
         [-23.3750, -23.7500, -12.1250,  ..., -22.1250, -22.8750, -21.2500],
         [-23.1250, -23.0000, -12.8750,  ..

In [ ]:
#Or
next_token_log_probas = -F.cross_entropy(
    logits[:, :-1].permute(0,2,1), next_token_ids, reduction="none"
)

In [9]:
[f"{p.item():.2%}" for p in torch.exp(next_token_log_probas[0])]

['3.11%', '0.02%', '0.01%', '3.11%', '0.38%', '32.42%', '16.41%', '98.44%']

In [10]:
[f"{p.item():.2%}" for p in torch.exp(next_token_log_probas[1])]

['3.11%', '0.02%', '0.01%', '3.11%', '0.38%', '32.42%', '0.00%', '0.00%']

In [12]:
token_index = 5
topk = torch.topk(log_probas[0, token_index], k=10)
for v, i in zip(topk.values, topk.indices):
    print(f"{torch.exp(v).item():06.2%} {mistral7b_tokenizer.decode([i])}")

32.42% is
30.47% ,
04.69% has
03.22% and
02.83% ,
02.50% .
02.36% Buenos
01.43% was
01.34% –
01.26% 



In [ ]:
answer_log_proba = next_token_log_probas[0, -2:].sum()  # Buenos + Aires
torch.exp(answer_log_proba).item()

0.162109375

In [21]:
next_token_log_probas.shape

torch.Size([2, 8, 1])

In [25]:
padding_mask = encodings.attention_mask[:, :-1]
log_probas_sum = (next_token_log_probas * padding_mask.unsqueeze(-1)).sum(dim=1)
log_probas_sum



tensor([[-33.2500],
        [-55.7500]], device='cuda:0', dtype=torch.bfloat16)

In [26]:
def sum_of_log_probas(model, tokenizer, full_inputs):
    encodings = tokenizer(
        full_inputs, return_tensors="pt", padding=True).to(model.device)
    logits = model(**encodings).logits
    next_token_log_probas = -F.cross_entropy(
        logits[:, :-1].permute(0, 2, 1), encodings.input_ids[:, 1:],
        reduction="none")
    return (next_token_log_probas * encodings.attention_mask[:, :-1]).sum(dim=1)

def dpo_loss(model, ref_model, tokenizer, full_input_c, full_input_r, beta=0.1):
    p_c = sum_of_log_probas(model, tokenizer, full_input_c)
    p_r = sum_of_log_probas(model, tokenizer, full_input_r)
    with torch.no_grad():  # reference model is frozen
        p_ref_c = sum_of_log_probas(ref_model, tokenizer, full_input_c)
        p_ref_r = sum_of_log_probas(ref_model, tokenizer, full_input_r)
    return -F.logsigmoid(beta*((p_c - p_ref_c) - (p_r - p_ref_r))).mean()



In [8]:
import shutil
from pathlib import Path

# Calea către cache-ul Hugging Face pentru acest model
cache_dir = Path.home() / ".cache" / "huggingface" / "hub" / "models--mistralai--Mistral-7B-Instruct-v0.3"

if cache_dir.exists():
    shutil.rmtree(cache_dir)
    print("Cache-ul corupt a fost șters cu succes!")
else:
    print("Folderul de cache nu a fost găsit în calea standard.")


Cache-ul corupt a fost șters cu succes!


In [12]:
model_id ="mistralai/Mistral-7B-Instruct-v0.3"
mistral7bi_tokenizer = AutoTokenizer.from_pretrained(model_id)
mistral7bi = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", dtype="auto"
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [13]:
good_bob = BobTheBot(mistral7bi, mistral7bi_tokenizer)
print(good_bob.chat("Tell me 5 jokes"))

Sure, here are five jokes for you:

1. Why don't scientists trust atoms?
   Because they make up everything!

2. Why did the scarecrow win an award?
   Because he was outstanding in his field!

3. Why did the bicycle fall over?
   Because it was two-tired!

4. Why did the computer go to therapy?
   Because it had a lot of data!

5. Why did the hipster burn his tongue?
   He drank his coffee before it was cool!

I hope you enjoyed these jokes! If you have any other requests, feel free to ask.


In [14]:
good_bob = BobTheBot(mistral7bi, mistral7bi_tokenizer)
print(good_bob.chat("How can I make cookies?"))

To make cookies, you'll need the following ingredients:
- 2 1/4 cups all-purpose flour
- 1 teaspoon baking soda
- 1 cup unsalted butter, softened
- 1/2 cup granulated sugar
- 1 cup packed light-brown sugar
- 1 teaspoon salt
- 2 teaspoons vanilla extract
- 2 large eggs
- 2 cups rolled oats
- 2 cups semisweet chocolate chips
- 1 1/2 cups chopped nuts (optional)

Preheat your oven to 350°F (175°C). In a small bowl, whisk together the flour and baking soda; set aside.

In a large bowl, beat the butter, granulated sugar, brown sugar, and salt with an electric mixer on medium speed until light and fluffy. Beat in the vanilla extract, then add the eggs, one at a time, beating well after each addition.

Reduce the mixer speed to low, gradually add the flour mixture, and mix just until combined. Stir in the oats, chocolate chips, and nuts (if using).

Drop rounded tablespoons of dough onto ungreased baking sheets, spacing them about 2 inches apart.

Bake the cookies until they are golden around